# Conditional Tasks

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) CrewAI roadmap topic **[Conditional Tasks](https://agent-foundry-pi.vercel.app/roadmaps/crewai/conditional-tasks)**.

You will learn how **ConditionalTask** runs only when a condition on the **previous** task’s output is `True`, how that behaves in a **sequential** crew, and how **guardrails** relate to regular tasks.

## 1. Install Dependencies

In [ ]:
!pip install -q crewai crewai-tools

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. ConditionalTask and a Full Pipeline

**ConditionalTask** executes **only** if the `condition` function returns `True` for the **previous** task’s `TaskOutput`; if it returns `False`, the task is **skipped**.

With **sequential** process, the **next** task after a skip still runs and receives the **last non-skipped** output as its prior context.

Minimal pattern (condition sees the **previous** task output):

```python
from crewai.tasks.conditional_task import ConditionalTask
from crewai.tasks.task_output import TaskOutput

def needs_more_data(output: TaskOutput) -> bool:
    return "insufficient" in output.raw.lower()

initial_task = Task(...)
fallback_task = ConditionalTask(
    description="Gather additional data from alternative sources",
    expected_output="Supplementary data",
    condition=needs_more_data,
    agent=researcher,
)
```

The runnable cell below wires **research → conditional deep-dive → analysis → report**.

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tasks.conditional_task import ConditionalTask
from crewai.tasks.task_output import TaskOutput

researcher = Agent(
    role="Research Analyst",
    goal="Gather accurate information on the topic",
    backstory="You research thoroughly and flag when data is thin or unclear.",
    verbose=True,
)

analyst = Agent(
    role="Analyst",
    goal="Turn research into structured insights",
    backstory="You synthesize findings into actionable conclusions.",
    verbose=True,
)

writer = Agent(
    role="Report Writer",
    goal="Produce a polished final report",
    backstory="You write concise, executive-ready prose.",
    verbose=True,
)


def needs_deep_dive(output: TaskOutput) -> bool:
    text = output.raw.lower()
    return any(
        phrase in text
        for phrase in ("insufficient", "unclear", "more research", "gaps in")
    )


research_task = Task(
    description="Research the assigned topic and summarize key facts and open questions.",
    expected_output="A structured summary; explicitly say if information is insufficient.",
    agent=researcher,
)

deep_dive_task = ConditionalTask(
    description="If prior output flagged gaps, dig into alternative angles and sources.",
    expected_output="Supplementary findings that address the gaps",
    condition=needs_deep_dive,
    agent=researcher,
)

analysis_task = Task(
    description="Synthesize all available research into themes, risks, and recommendations.",
    expected_output="Bullet analysis: themes, risks, 3–5 recommendations",
    agent=analyst,
)

report_task = Task(
    description="Write a short executive report from the analysis.",
    expected_output="Under 400 words, professional tone, sections: Summary, Findings, Recommendations",
    agent=writer,
)

crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_task, deep_dive_task, analysis_task, report_task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff()
print(result)

## 4. Task Guardrails (Related Concept)

**Guardrails** constrain how a regular **`Task`** should answer (length, tone, policy). They do **not** skip the task; they guide the agent. Use **ConditionalTask** when a step should run **only sometimes** based on prior output.

In [ ]:
from crewai import Agent, Task

writer = Agent(
    role="Writer",
    goal="Produce concise professional copy",
    backstory="You write for executives.",
    verbose=True,
)

task = Task(
    description="Summarize the main takeaway in one paragraph.",
    expected_output="One short paragraph.",
    agent=writer,
    guardrail="Output must be under 200 words and professional in tone",
)

print(task.description)

## Key Takeaways

- **ConditionalTask** runs **only** if `condition` returns **`True`** for the **previous** task’s output; otherwise it is **skipped**.
- This enables **dynamic workflows** that adapt to intermediate results.
- In **sequential** process, a skipped conditional does not block later tasks; the next task uses the **last non-skipped** output.
- **Guardrails** add **output rules** to a normal **`Task`**; they complement but do not replace conditional branching.